<a href="https://colab.research.google.com/github/arnav-singh-20/Arnav-Singh/blob/main/Lawgorithm_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
!cp "/content/drive/MyDrive/Lawgorithm/lawgorithm_train.jsonl" /content/


In [ ]:
!pip install -q -U transformers accelerate peft datasets bitsandbytes sentencepiece safetensors


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 140.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 511.6/511.6 kB 43.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 19.6 MB/s eta 0:00:00


In [ ]:
from datasets import load_dataset

dataset = load_dataset("json", data_files="lawgorithm_train.jsonl", split="train")
dataset = dataset.select(range(500))

def format_row(ex):
    task = ex.get("task", "")
    inp = ex.get("input", "")
    out = ex.get("output", ex.get("label", ""))

    return {
        "text": f"### Task: {task}\nInput:\n{inp}\n\nOutput:\n{out}"
    }

dataset = dataset.map(format_row)


from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "mistralai/Mistral-7B-Instruct-v0.2",
    use_fast=True
)
tokenizer.pad_token = tokenizer.eos_token

def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=512
    )

dataset = dataset.map(tokenize, batched=True)


dataset = dataset.remove_columns(
    [col for col in dataset.column_names if col not in ["input_ids", "attention_mask"]]
)

print(dataset)



Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 500
})


In [ ]:
# --------------------------
# 1. Load model in 4-bit
# --------------------------
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype="float16",
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto"
)

# --------------------------
# 2. Prepare for LoRA training
# --------------------------
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()


config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/4.94G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/4.54G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

trainable params: 13,631,488 || all params: 7,255,363,584 || trainable%: 0.1879


In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

training_args = TrainingArguments(
    output_dir="lawgorithm-500",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    save_steps=200,
    report_to="none",   # disable W&B if you want
)

collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,          # must contain ONLY input_ids+attention_mask
    data_collator=collator,
    processing_class=tokenizer      # replaces deprecated tokenizer=
)

trainer.train()


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
10,1.651200
20,1.430400
30,1.349000
40,1.217900
50,1.293000
60,1.277200
70,1.271300
80,1.205500
90,1.271600
100,1.306400


TrainOutput(global_step=125, training_loss=1.3066575736999513, metrics={'train_runtime': 926.907, 'train_samples_per_second': 0.539, 'train_steps_per_second': 0.135, 'total_flos': 1.0942911873024e+16, 'train_loss': 1.3066575736999513, 'epoch': 1.0})

In [ ]:
trainer.save_model("lawgorithm-500")
tokenizer.save_pretrained("lawgorithm-500")

print("MODEL SAVED!")


MODEL SAVED!


In [ ]:
trainer.save_model("lora_output")
tokenizer.save_pretrained("lora_output")


('lora_output/tokenizer_config.json',
 'lora_output/special_tokens_map.json',
 'lora_output/chat_template.jinja',
 'lora_output/tokenizer.model',
 'lora_output/added_tokens.json',
 'lora_output/tokenizer.json')

In [ ]:
!pip install bitsandbytes accelerate transformers peft -q


In [ ]:
!pip install -U bitsandbytes accelerate transformers peft -q


In [ ]:
!nvidia-smi


Tue Dec  2 22:54:53 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   73C    P0             33W /   70W |    7268MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

BASE = "mistralai/Mistral-7B-Instruct-v0.2"
LORA = "/content/drive/MyDrive/Lawgorithm/lora_output"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(BASE)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE,
    quantization_config=bnb_config,
    device_map="auto"
)

print("Mistral Loaded in 4-bit!")


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Mistral Loaded in 4-bit!


In [ ]:
!cp -r /content/lawgorithm-500 "/content/drive/MyDrive/Lawgorithm/"


In [ ]:
import os

LORA = "/content/drive/MyDrive/Lawgorithm/lawgorithm-500/lora_output"
print(os.listdir(LORA))


['README.md', 'adapter_model.safetensors', 'adapter_config.json', 'chat_template.jinja', 'tokenizer_config.json', 'special_tokens_map.json', 'tokenizer.model', 'tokenizer.json', 'training_args.bin']


In [ ]:
from peft import PeftModel

model = PeftModel.from_pretrained(
    base_model,
    LORA,
    device_map="auto"
)

model.eval()
print("LoRA Loaded!")


LoRA Loaded!


In [ ]:
def ask(clause):
    prompt = f"""
You are a legal assistant. Complete the following template. Do NOT output anything extra.

Clause:
{clause}

Summary:
"""

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=120,
        temperature=0.0,   # FORCE deterministic output
        do_sample=False,   # NO sampling
        eos_token_id=tokenizer.eos_token_id
    )

    print(tokenizer.decode(outputs[0], skip_special_tokens=True))


In [ ]:
def ask(clause, task="summary"):

    if task == "summary":
        field = "Summary"
        instruction = "Provide a short summary of the clause."

    elif task == "risk":
        field = "Risks"
        instruction = "List potential legal risks in the clause."

    elif task == "explain":
        field = "Explanation"
        instruction = "Explain the clause in simple language."

    elif task == "recommend":
        field = "Recommendations"
        instruction = "Provide recommendations to improve the clause."

    else:
        raise ValueError("Invalid task")

    prompt = f"""
You are a legal assistant. {instruction}

Clause:
{clause}

{field}:
"""

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        temperature=0.0,       # deterministic
        do_sample=False,
        eos_token_id=tokenizer.eos_token_id
    )

    print(tokenizer.decode(outputs[0], skip_special_tokens=True))


In [ ]:
ask("The landlord may terminate the agreement anytime without notice.")


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.



You are a legal assistant. Provide a short summary of the clause.

Clause:
The landlord may terminate the agreement anytime without notice.

Summary:
The landlord has the right to terminate the agreement at any time without providing notice to the tenant.


In [ ]:
ask( "The tenant must pay rent by the 10th. Late payment attracts a 5% penalty.")


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.



You are a legal assistant. Provide a short summary of the clause.

Clause:
The tenant must pay rent by the 10th. Late payment attracts a 5% penalty.

Summary:
The tenant is required to pay rent by the 10th of each month. Failure to pay on time will result in a 5% penalty.


In [ ]:
ask("The app may share user location data with third parties for service improvement.")


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.



You are a legal assistant. Provide a short summary of the clause.

Clause:
The app may share user location data with third parties for service improvement.

Summary:
The app may share user location data with third parties to improve the services.


In [ ]:
def ask_simplify(text):
    prompt = f"""
You are a legal assistant. Rewrite the following clause in simple, easy-to-understand language while preserving its legal meaning.

Clause:
{text}

Simplified Version:
"""

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    output = model.generate(
        **inputs,
        max_new_tokens=150,
        temperature=0.0,
        do_sample=False,
        eos_token_id=tokenizer.eos_token_id,
    )
    print(tokenizer.decode(output[0], skip_special_tokens=True))


In [ ]:
ask_simplify("The indemnified party shall be held harmless from all liabilities arising from the indemnifying party’s negligent acts or omissions.")


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.



You are a legal assistant. Rewrite the following clause in simple, easy-to-understand language while preserving its legal meaning.

Clause:
The indemnified party shall be held harmless from all liabilities arising from the indemnifying party’s negligent acts or omissions.

Simplified Version:
The party being protected (indemnified party) won’t be responsible for any damages caused by the other party’s (indemnifying party) carelessness.
